###Save all .parquet files data from landing container to bronze schema in table format


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import *  
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("landing_to_bronze")

catalog='catalog_dev'
source_schema='landing'
target_schema='bronze'
storage_account='onprimtocloudstorgaeacc'


spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {source_schema}")

DataFrame[]

In [0]:
# function to create dataframe

def create_DF(file):
    df=spark.read.format('parquet').load(file)
    return df


In [0]:
# function to save dataframe as table in Unity Catalog

def create_table(df,table):
    target_path=f"abfss://{target_schema}@{storage_account}.dfs.core.windows.net/{table}"
    df.write.format('delta') \
            .mode('overwrite')\
            .option('path', target_path) \
            .option("overwriteSchema", "true")\
            .saveAsTable(f'{catalog}.{target_schema}.{table} ')

In [0]:
import os

files=[]
for file in dbutils.fs.ls('/Volumes/catalog_dev/landing/saleslt/'):
    files=dbutils.fs.ls(file.path)
    data_file=[]
    
    # create dataframes
    for data_file in files:
        df=create_DF(data_file.path) 
        
        #saving table into bronze schema
        table_name=os.path.splitext(data_file.path.split('/')[-1])[0]
        logger.info(f"Saving {catalog}.{target_schema}.{table_name} ")
        create_table(df,table_name)
        

2026-06-04 07:43:45,273 - INFO - Saving catalog_dev.bronze.Address 
2026-06-04 07:43:56,591 - INFO - Saving catalog_dev.bronze.Customer 
2026-06-04 07:44:07,264 - INFO - Saving catalog_dev.bronze.CustomerAddress 
2026-06-04 07:44:18,702 - INFO - Saving catalog_dev.bronze.Product 
2026-06-04 07:44:30,515 - INFO - Saving catalog_dev.bronze.ProductCategory 
2026-06-04 07:44:41,824 - INFO - Saving catalog_dev.bronze.ProductDescription 
2026-06-04 07:44:52,906 - INFO - Saving catalog_dev.bronze.ProductModel 
2026-06-04 07:45:03,926 - INFO - Saving catalog_dev.bronze.ProductModelProductDescription 
2026-06-04 07:45:15,021 - INFO - Saving catalog_dev.bronze.SalesOrderDetail 
2026-06-04 07:45:26,170 - INFO - Saving catalog_dev.bronze.SalesOrderHeader 
